<a href="https://colab.research.google.com/github/tfanmd/internship-progress/blob/main/finetune_indobart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Jalankan di CELL 1 (Versi Downgrade Stabil)
!pip install -q transformers==4.40.0 datasets accelerate==0.30.0 evaluate sacrebleu chrf jiwer nltk rouge_score

import nltk
nltk.download('wordnet')
nltk.download('punkt')

print("✓ Environment Berhasil Di-downgrade ke Versi Stabil!")

ERROR: Could not find a version that satisfies the requirement chrf (from versions: none)
ERROR: No matching distribution found for chrf
✓ Environment Berhasil Di-downgrade ke Versi Stabil!


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [2]:
# Jalankan di Cell 2
import pandas as pd
from datasets import Dataset, DatasetDict

train_df = pd.read_csv("/content/test.csv")
val_df = pd.read_csv("/content/val.csv")
test_df = pd.read_csv("/content/test.csv")

# Sesuai fokus lu, kita buat kolom tiruan 'source' agar script evaluasi tetap jalan seragam
test_df['source'] = 'Combined'

raw_datasets = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
    "test": Dataset.from_pandas(test_df)
})

print(f"✓ Dataset IndoBART-v2 sukses dimuat! Jumlah data uji: {len(raw_datasets['test'])}")

✓ Dataset IndoBART-v2 sukses dimuat! Jumlah data uji: 1962


In [2]:
# Jalankan di CELL 3 (Versi Perbaikan Ukuran Matriks Embedding)
from transformers import MBartTokenizerFast, MBartForConditionalGeneration, GenerationConfig

model_name = "indobenchmark/indobart-v2"

print("=== MEMULAI LOAD MODEL & TOKENIZER INDOBART-V2 ===")

# 1. Load Tokenizer
tokenizer = MBartTokenizerFast.from_pretrained(model_name)

# 2. Load Model Sequence-to-Sequence
model = MBartForConditionalGeneration.from_pretrained(model_name)

# 🎯 SOLUSI UTAMA EROR CUDA: Paksa model menyesuaikan ukuran kosakata tokenizer!
model.resize_token_embeddings(len(tokenizer))

# 3. Ikat parameter generasi teks agar aman di library versi baru
generation_config = GenerationConfig(
    max_length=64,
    min_length=3,
    no_repeat_ngram_size=3,
    early_stopping=True,
    length_penalty=2.0,
    num_beams=4,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id
)
model.generation_config = generation_config

print(f"\n✓ Vocab Size Tokenizer: {len(tokenizer)}")
print("✓ Arsitektur IndoBART-v2 & Tokenizer Sukses Disinkronisasikan!")

=== MEMULAI LOAD MODEL & TOKENIZER INDOBART-V2 ===


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/267 [00:00<?, ?it/s]

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`



✓ Vocab Size Tokenizer: 40026
✓ Arsitektur IndoBART-v2 & Tokenizer Sukses Disinkronisasikan!


In [5]:
# Jalankan di CELL 4
import random

print("=== MEMULAI TOKENISASI DATASET INDOBART-V2 ===")

# Ganti isi fungsi di CELL 4 dengan ini ya, bro!
def preprocess_function(examples):
    inputs = examples["indonesia"]
    targets = examples["jawa"]

    # 🎯 KUNCI UTAMA: Set penanda bahasa Indonesia (id_ID) agar huruf depan gak hilang
    tokenizer.src_lang = "id_ID"
    tokenizer.tgt_lang = "id_ID" # Ikut gunakan id_ID agar pemetaan token target tetap utuh

    # Proses tokenisasi input dan target sekaligus dalam satu baris aman
    model_inputs = tokenizer(
        inputs,
        text_target=targets,      # 🎯 SOLUSI BARU: Pakai text_target langsung, bypass as_target_tokenizer!
        max_length=64,
        truncation=True,
        padding="max_length"
    )

    return model_inputs

# Proses tokenisasi secara paralel pada seluruh data
tokenized_datasets = raw_datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=raw_datasets["train"].column_names
)

print("✓ Tokenisasi seluruh dataset selesai!")
print("\n========================================================")
print("       PEMANTAUAN CONTOH HASIL TOKENISASI (SANITY CHECK) ")
print("========================================================")

# Ambil satu indeks acak dari data train untuk contoh display
idx_contoh = random.randint(0, len(raw_datasets["train"]) - 1)

teks_indo_asli = raw_datasets["train"][idx_contoh]["indonesia"]
teks_jawa_asli = raw_datasets["train"][idx_contoh]["jawa"]

token_input_ids = tokenized_datasets["train"][idx_contoh]["input_ids"]
token_label_ids = tokenized_datasets["train"][idx_contoh]["labels"]

# Decode kembali token ID angka ke teks untuk pembuktian
decode_indo = tokenizer.decode(token_input_ids, skip_special_tokens=True)
decode_jawa = tokenizer.decode(token_label_ids, skip_special_tokens=True)

print(f"👉 [Indeks Data]: {idx_contoh}")
print(f"➡️ [Teks Indonesia Asli] : {teks_indo_asli}")
print(f"🔢 [Token IDs Input]     : {token_input_ids[:15]}... (dipotong dari total 64)")
print(f"🔄 [Hasil Decode Input]  : {decode_indo}\n")

print(f"➡️ [Teks Jawa Asli]      : {teks_jawa_asli}")
print(f"🔢 [Token IDs Target]    : {token_label_ids[:15]}... (dipotong dari total 64)")
print(f"🔄 [Hasil Decode Target] : {decode_jawa}")
print("========================================================")

=== MEMULAI TOKENISASI DATASET INDOBART-V2 ===


Map:   0%|          | 0/1962 [00:00<?, ? examples/s]

Map:   0%|          | 0/1961 [00:00<?, ? examples/s]

Map:   0%|          | 0/1962 [00:00<?, ? examples/s]

✓ Tokenisasi seluruh dataset selesai!

       PEMANTAUAN CONTOH HASIL TOKENISASI (SANITY CHECK) 
👉 [Indeks Data]: 1854
➡️ [Teks Indonesia Asli] : Dia penulis lebih dari 250 kertas kerja ilmiah begitu juga sebuah buku The Conquest of Fertility, tentang penaklukan kesuburan, terbit tahun 1965.
🔢 [Token IDs Input]     : [39936, 3, 382, 267, 263, 368, 300, 506, 262, 337, 323, 3220, 268, 264, 3403]... (dipotong dari total 64)
🔄 [Hasil Decode Input]  : ia penulis lebih dari 250 kertas kerja ilmiah begitu juga sebuah buku he onquest of ertility, tentang penaklukan kesuburan, terbit

➡️ [Teks Jawa Asli]      : Panjenenganipun sampun nyerat langkung saking 250 kertas kerja ilmiah lan buku The Eggs of Mammals, buku Conquest of Fertility, taun 1965.
🔢 [Token IDs Target]    : [39936, 3, 261, 2282, 307, 261, 511, 279, 265, 449, 279, 1019, 264, 273, 299]... (dipotong dari total 64)
🔄 [Hasil Decode Target] : anjenenganipun sampun nyerat langkung saking 250 kertas kerja ilmiah lan buku he ggs of ammal

In [6]:
# Jalankan di Cell 5
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
import torch

torch.cuda.empty_cache()

training_args = Seq2SeqTrainingArguments(
    output_dir="./results_indobart_translator",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=4,    # Batch 4 masih tergolong aman untuk IndoBART di T4
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,    # Total effective batch size = 16
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=5,
    predict_with_generate=True,
    fp16=True,                        # Mixed precision agar hemat memori
    logging_steps=50,
    load_best_model_at_end=True,
    report_to="none",
    generation_max_length=64,
    generation_num_beams=4
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer         # Menggunakan standarisasi 'processing_class' terbaru
)

print("✓ Trainer IndoBART-v2 dikunci aman dan siap melaju!")

✓ Trainer IndoBART-v2 dikunci aman dan siap melaju!


In [7]:
# Jalankan di Cell 6
print("=== MEMULAI PROSES TRAINING INDOBART-V2 SUITE ===")
trainer.train()
print("\n✓ Training selesai! Model terbaik otomatis dimuat ke memori.")

=== MEMULAI PROSES TRAINING INDOBART-V2 SUITE ===


Epoch,Training Loss,Validation Loss
1,10.658704,2.391238
2,8.977517,2.131008
3,8.045266,2.017364
4,7.348930,1.972030
5,7.179279,1.955215


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].



✓ Training selesai! Model terbaik otomatis dimuat ke memori.


In [11]:
%pip install evaluate sacrebleu chrf jiwer rouge_score

  Using cached evaluate-0.4.6-py3-none-any.whl.metadata (9.5 kB)
  Using cached sacrebleu-2.6.0-py3-none-any.whl.metadata (39 kB)
ERROR: Could not find a version that satisfies the requirement chrf (from versions: none)
ERROR: No matching distribution found for chrf


In [12]:
# Jalankan di Cell 7
import evaluate
import pandas as pd
import torch
from tqdm import tqdm

print("=== MEMULAI EVALUASI AKHIR TOTAL (INDOBART-V2) ===")
model.eval()

test_inputs = raw_datasets["test"]["indonesia"]
test_references = raw_datasets["test"]["jawa"]
test_predictions = []

# Load metrik standar dosen pembimbing
bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")
meteor_metric = evaluate.load("meteor")
ter_metric = evaluate.load("ter")

print(f"Menerjemahkan {len(test_inputs)} total data uji...")

batch_size = 4
for i in tqdm(range(0, len(test_inputs), batch_size)):
    batch_texts = test_inputs[i:i+batch_size]
    inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=64).to("cuda")

    with torch.no_grad():
        generated_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=64,
            num_beams=4
        )
    decoded_preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    test_predictions.extend([pred.strip() for pred in decoded_preds])

print("\n=== MENGHITUNG METRIK EVALUASI UTUH ===")
formatted_references_bleu = [[ref.strip()] for ref in test_references]
formatted_references_rest = [[ref.strip()] for ref in test_references]

bleu_results = bleu_metric.compute(predictions=test_predictions, references=formatted_references_bleu)
chrf_results = chrf_metric.compute(predictions=test_predictions, references=formatted_references_rest)
meteor_results = meteor_metric.compute(predictions=test_predictions, references=test_references)
ter_results = ter_metric.compute(predictions=test_predictions, references=formatted_references_bleu)

print("\n========================================================")
print("     HASIL TESTING AKHIR TOTAL MODEL (INDOBART-V2)      ")
print("\n========================================================")
print(f" Skor BLEU Akhir   : {round(bleu_results['score'], 2)}")
print(f" Skor ChrF Akhir   : {round(chrf_results['score'], 2)}")
print(f" Skor METEOR Akhir : {round(meteor_results['meteor'] * 100, 2)}")
print(f" Skor TER Akhir    : {round(ter_results['score'], 2)}  <-- (Makin KECIL makin BAGUS)")
print("========================================================\n")

df_hasil = pd.DataFrame({
    "Teks Indonesia (Input)": test_inputs,
    "Teks Jawa Asli (Target)": test_references,
    "Hasil Terjemahan IndoBART (Prediksi)": test_predictions
})
df_hasil.to_csv("hasil_testing_indobart_total.csv", index=False, encoding="utf-8")
print("✓ Berkas 'hasil_testing_indobart_total.csv' sukses disimpan!")

ModuleNotFoundError: No module named 'evaluate'

In [3]:
# JALANKAN DI CELL 3 (FULL CODE)
from transformers import MBartTokenizerFast, MBartForConditionalGeneration, GenerationConfig

model_path = "./model_indobart_terbaik"

print("=== LOADING MODEL DARI FOLDER LOKAL ===")

# 1. Load Tokenizer dari folder lokal
tokenizer = MBartTokenizerFast.from_pretrained(model_path)

# 2. Load Model dari folder lokal
model = MBartForConditionalGeneration.from_pretrained(model_path)

# 3. Sinkronisasikan ukuran token embeddings
model.resize_token_embeddings(len(tokenizer))

# 4. Ikat konfigurasi generasi teks demi kelancaran proses testing
generation_config = GenerationConfig(
    max_length=64,
    min_length=3,
    no_repeat_ngram_size=3,
    early_stopping=True,
    length_penalty=2.0,
    num_beams=4,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id
)
model.generation_config = generation_config

print("\n✓ Model pinter hasil latihan lu berhasil dimuat dan dikunci!")

=== LOADING MODEL DARI FOLDER LOKAL ===


Loading weights:   0%|          | 0/264 [00:00<?, ?it/s]


✓ Model pinter hasil latihan lu berhasil dimuat dan dikunci!


In [7]:
# JALANKAN DI CELL 7 (VERSI FIX CUDA SYNC DEVICE)
import sys

print("=== FIXING PACKAGES & MEMULAI EVALUASI AKHIR TOTAL ===")

# 1. Pastikan library metrik terpasang bersih tanpa typo package siluman
import subprocess
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "evaluate", "sacrebleu", "jiwer", "rouge_score"])
    print("✓ Semua library metrik sukses terpasang sempurna!")
except Exception as e:
    print(f"Peringatan instalasi internal: {e}")

import pandas as pd
import torch
from tqdm import tqdm
import evaluate

# 🎯 SOLUSI UTAMA EROR: Lempar fisik objek model lu dari CPU ke GPU (CUDA)
if torch.cuda.is_available():
    model = model.to("cuda")
    print("✓ Model IndoBART-v2 sukses dipindahkan ke GPU (CUDA)!")
else:
    print("⚠️ Peringatan: GPU tidak terdeteksi, berjalan di CPU (Akan Lambat)")

model.eval()

test_inputs = raw_datasets["test"]["indonesia"]
test_references = raw_datasets["test"]["jawa"]
test_predictions = []

# Load metrik resmi Hugging Face
bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")
meteor_metric = evaluate.load("meteor")
ter_metric = evaluate.load("ter")

print(f"\nMenerjemahkan {len(test_inputs)} total data uji... (Harap tunggu)")

# 2. Proses Penerjemahan Paralel di GPU
batch_size = 4
for i in tqdm(range(0, len(test_inputs), batch_size)):
    batch_texts = test_inputs[i:i+batch_size]
    # Teks input dikirim ke CUDA
    inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=64).to("cuda")

    with torch.no_grad():
        generated_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=64,
            num_beams=4
        )
    decoded_preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    test_predictions.extend([pred.strip() for pred in decoded_preds])

print("\n=== MENGHITUNG METRIK EVALUASI UTUH ===")
formatted_references_bleu = [[ref.strip()] for ref in test_references]
formatted_references_rest = [[ref.strip()] for ref in test_references]

# 3. Kalkulasi Skor Akhir
bleu_results = bleu_metric.compute(predictions=test_predictions, references=formatted_references_bleu)
chrf_results = chrf_metric.compute(predictions=test_predictions, references=formatted_references_rest)
meteor_results = meteor_metric.compute(predictions=test_predictions, references=test_references)
ter_results = ter_metric.compute(predictions=test_predictions, references=formatted_references_bleu)

print("\n========================================================")
print("     HASIL TESTING AKHIR TOTAL MODEL (INDOBART-V2)      ")
print("\n========================================================")
print(f" Skor BLEU Akhir   : {round(bleu_results['score'], 2)}")
print(f" Skor ChrF Akhir   : {round(chrf_results['score'], 2)}")
print(f" Skor METEOR Akhir : {round(meteor_results['meteor'] * 100, 2)}")
print(f" Skor TER Akhir    : {round(ter_results['score'], 2)}  <-- (Makin KECIL makin BAGUS)")
print("========================================================\n")

# 4. Simpan Hasil ke CSV
df_hasil = pd.DataFrame({
    "Teks Indonesia (Input)": test_inputs,
    "Teks Jawa Asli (Target)": test_references,
    "Hasil Terjemahan IndoBART (Prediksi)": test_predictions
})
df_hasil.to_csv("hasil_testing_indobart_total.csv", index=False, encoding="utf-8")
print("✓ Berkas 'hasil_testing_indobart_total.csv' sukses disimpan!")

=== FIXING PACKAGES & MEMULAI EVALUASI AKHIR TOTAL ===
✓ Semua library metrik sukses terpasang sempurna!
✓ Model IndoBART-v2 sukses dipindahkan ke GPU (CUDA)!


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!



Menerjemahkan 1962 total data uji... (Harap tunggu)


100%|██████████| 491/491 [05:58<00:00,  1.37it/s]



=== MENGHITUNG METRIK EVALUASI UTUH ===

     HASIL TESTING AKHIR TOTAL MODEL (INDOBART-V2)      

 Skor BLEU Akhir   : 6.7
 Skor ChrF Akhir   : 32.76
 Skor METEOR Akhir : 20.05
 Skor TER Akhir    : 86.47  <-- (Makin KECIL makin BAGUS)

✓ Berkas 'hasil_testing_indobart_total.csv' sukses disimpan!


In [13]:
trainer.save_model("./model_indobart_terbaik")
tokenizer.save_pretrained("./model_indobart_terbaik")
print("✓ Aman! Model terbaik lu sudah diselamatkan ke folder.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Aman! Model terbaik lu sudah diselamatkan ke folder.


In [8]:
import shutil

# Kompres folder lokal model_indobart_terbaik menjadi file zip
shutil.make_archive("model_indobart", 'zip', "./model_indobart_terbaik")
print("✓ Folder model berhasil dikompres menjadi 'model_indobart.zip'!")

✓ Folder model berhasil dikompres menjadi 'model_indobart.zip'!


In [12]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
import shutil
import os

# Tentukan jalur tujuan di dalam Google Drive lu (Folder Utama / My Drive)
tujuan_drive = "/content/drive/MyDrive/model_indobart_terbaik"

# Proses pemindahan file mentah langsung ke Drive cloud lu
if os.path.exists("./model_indobart_terbaik"):
    shutil.copytree("./model_indobart_terbaik", tujuan_drive, dirs_exist_ok=True)
    print("✓ BERHASIL! Folder model lu sekarang sudah aman pindah ke Google Drive!")
else:
    print("⚠️ Folder model lokal tidak ditemukan. Coba cek panel kiri apakah namanya sesuai.")

✓ BERHASIL! Folder model lu sekarang sudah aman pindah ke Google Drive!
